# Split Views, Layouts, and Control Views

**Part I · Visualization** — Tutorial 14

Compose multiple scenes and control panels into a single browser page with the
`View` hierarchy. You will learn to:

- Arrange panes with `SplitView` (nestable splits, draggable splitters) and
  `StackView` (vertical/horizontal/wrap flow).
- Render a scene in a `SceneView` (with overlays and a per-pane camera) and
  group controls with `GroupView`.
- Use the HTML control views (`SliderView`, `DropdownView`, `ButtonView`,
  `ValueEditView`, `FileChooserView`, …).
- Open a layout with `show(layout=...)` and re-aim a pane with
  `set_view_camera`.


## Setup


In [ ]:
from pytanga.geometry import Point, Sphere
from pytanga.viz import (
    ButtonView, CameraConfig3d, GroupView, SceneView, Size, SliderView,
    SplitView, StackView, Visualizer,
)


## 1. The `View` hierarchy

| Class | Purpose |
|---|---|
| `View` | base for every pane/container (per-axis sizes) |
| `SplitView` | container laying children along one axis with draggable splitters |
| `StackView` | flex container stacking children vertically / horizontally / wrapping |
| `SceneView` | a pane rendering a named scene, with overlays and an initial camera |
| `GroupView` | a titled control group (pane or scene overlay) |
| `SpacerView` | an empty, fully-flexible filler |
| `SliderView` / `ButtonView` / `DropdownView` / … | a single HTML control as a `View` |

> Text / color / checkbox view counterparts (`TextFieldView`, `TextAreaView`,
> `ColorPickerView`, `CheckboxView`) live in `pytanga.viz.views`.


## 2. `SplitView` + `SceneView`

`SplitView(orientation, children)` splits along one axis; `SceneView(scene)`
renders a scene (`""` is the main scene, a name or handle for others).


In [ ]:
viz = Visualizer(reuse_existing=False, title="Split view — two scenes")

viz.add(Sphere(Point(0, 0, 0), 2.0), color="#4488ff", opacity=0.3)
side = viz.scene("side")
side.add(Point(2, 0, 0), color="#44ff44")

layout = SplitView(
    orientation="horizontal",
    children=[SceneView(""), SceneView("side")],
)

# viz.show(layout=layout)   # opens both panes at a single URL
viz.set_layout(layout)
print("layout registered:", viz.url)


## 3. `StackView`, `GroupView`, and control views

`StackView` stacks children in normal flow (no splitters); `GroupView` adds a
titled chrome. Control views (`SliderView`, `ButtonView`, …) render one HTML
control each, with handlers registered automatically when the layout is set.


In [ ]:
async def on_reset(_value, _event):
    print("reset")

viz = Visualizer(reuse_existing=False, title="Split view — controls")
viz.add(Sphere(Point(0, 0, 0), 2.0), opacity=0.3)

sidebar = GroupView(
    "Actions",
    [
        SliderView("radius", label="Radius", min=0.1, max=5.0, value=2.0),
        ButtonView("btn_fit", label="Fit camera"),
        StackView("vertical", [ButtonView("a", label="A"), ButtonView("b", label="B")]),
    ],
)
layout = SplitView("horizontal", [sidebar, SceneView("")])
viz.set_layout(layout)
print("control layout registered")


## 4. Sizes and units

Every `View` accepts per-axis constraints via `Size` values:

| Unit | Meaning | API |
|---|---|---|
| `px` | absolute CSS pixels | `Size.px(280)` |
| `%` | fraction of the parent extent | `Size.percent(50)` |
| `fr` | flexible share (preferred only) | `Size.fr(2)` |
| `auto` | unconstrained | `Size.auto()` |

A view is **fixed** along an axis when its `min` and `max` are equal; a
splitter is draggable only when both neighbours are non-fixed along the split
axis.


## 5. Per-pane camera and runtime `set_view_camera`

`SceneView(scene, camera=…)` gives one pane a different **initial** camera.
`set_view_camera(view, camera)` re-aims a single pane at runtime.


In [ ]:
viz = Visualizer(reuse_existing=False, title="Split view — per-pane camera")
viz.add(Sphere(Point(0, 0, 0), 2.0), opacity=0.3)

top = SceneView("", camera=CameraConfig3d(position=(0, 0, 8), target=(0, 0, 0)))
side = SceneView("", camera=CameraConfig3d(position=(8, 0, 0), target=(0, 0, 0)))

layout = SplitView("horizontal", [top, side])
viz.set_layout(layout)

# Re-aim one pane at runtime (targeted by the SceneView instance):
viz.set_view_camera(top, CameraConfig3d(position=(0, 8, 0), target=(0, 0, 0)))
print("per-pane camera set")


## 6. Opening a layout

`show(layout=...)` (or `run(layout=...)`) registers the layout and opens it at
one URL (`/?view=<name>`). A browser subscribes to every pane's scene over a
single WebSocket connection.


## Visual Examples

A nested split-view layout (control sidebar + main scene + a second scene) —
the same structure as the library's `split_view.py` demo.


In [ ]:
viz = Visualizer(reuse_existing=False, title="Split view — demo")

viz.add(Sphere(Point(0, 0, 0), 2.0), color="#4488ff", opacity=0.3)
side = viz.scene("side")
side.add(Point(2, 0, 0), color="#44ff44")

main_view = SceneView("")
layout = SplitView(
    orientation="horizontal",
    children=[
        GroupView(
            "Actions",
            [
                SliderView("radius", label="Radius", min=0.1, max=5.0, value=2.0),
                ButtonView("btn_fit", label="Fit camera"),
                ButtonView("btn_topdown", label="Top-down"),
            ],
        ),
        SplitView(
            orientation="vertical",
            sizes=[Size.percent(60), Size.percent(40)],
            children=[
                main_view,
                SplitView(
                    orientation="horizontal",
                    children=[
                        SceneView("side"),
                        SceneView("", camera=CameraConfig3d(position=(8, 0, 0), target=(0, 0, 0))),
                    ],
                ),
            ],
        ),
    ],
)

viz.show(layout=layout)   # opens the full layout at a single URL
print("split view layout opened")


## Summary

| Task | API |
|---|---|
| Horizontal / vertical split | `SplitView("horizontal" | "vertical", [children])` |
| Stack / wrap | `StackView("vertical" | "horizontal" | "wrap", [children])` |
| Scene pane | `SceneView("scene_name", camera=..., overlay=[...])` |
| Control group | `GroupView("title", [controls])` |
| Filler | `SpacerView()` |
| Sizes | `Size.px(n)` / `Size.percent(n)` / `Size.fr(n)` / `Size.auto()` |
| Control view | `SliderView(cid, label, min, max, value, on_change)` |
| Open a layout | `show(layout=...)` / `run(layout=...)` |
| Re-aim a pane | `set_view_camera(scene_view, camera)` |

**Next:** [15 — VisualizerApp](../15_visualizer_app/).
